#  Generation Pipleline - The "G" in RAG

We are going to work on:-

Question → Retrieve relevant chunks → Build a prompt →  Send to Groq(Model) → Get a GROUNDED ANSWER

We are connecting the __"R"__(Retrieval) you built (file name is _02_embeddings.ipynb_) to the __"G"__ (Generation) — completing the full RAG loop for the first time.



rag_query("your question")                        
                                                    
1. retrieve_chunks()  → ChromaDB semantic search  
2. build_prompt()     → combine context+question  
3. generate_answer()  → Groq LLaMA generates      
                                                   
Returns: grounded, accurate answer                


In [1]:
import sys
print("Python:", sys.executable)

from groq import Groq
from google import genai
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from pathlib import Path
import os

env_path = Path(r"C:\Users\jasme\Documents\rag-knowledge-assistant\.env")
load_dotenv(dotenv_path=env_path, override=True)

# Reconnect to your persistent ChromaDB
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

persistent_client = chromadb.PersistentClient(
    path=r"C:\Users\jasme\Documents\rag-knowledge-assistant\chroma_db"
)

collection = persistent_client.get_collection(
    name="rag_paper",
    embedding_function=sentence_transformer_ef
)

print(f"✅ Connected! {collection.count()} chunks ready")


Python: C:\Users\jasme\anaconda3\python.exe


C:\Users\jasme\anaconda3\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3755.27it/s]


✅ Connected! 93 chunks ready


In [2]:
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))
print("✅ Groq client ready")
print("\nAll systems go — starting!")

✅ Groq client ready

All systems go — starting!


In [3]:
genai_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
print("✅ GEMINI client ready")
print("\nAll systems go — starting!")

✅ GEMINI client ready

All systems go — starting!


__Step 1__ — Write the RETRIEVAL function (wrap pervious working file)

In [4]:
def retrieve_chunks(query, n_results=3):
    """
    
    Takes a question, returns the most relevant chunks from the PDF.
    
    """
    
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    return results['documents'][0]

# Quick test
test_chunks = retrieve_chunks("What challenges does this paper address with PDFs?")

print(f"Retrieved {len(test_chunks)} chunks")
print(f"\nFirst chunk preview:\n{test_chunks[0][:200]}...")

Retrieved 3 chunks

First chunk preview:
academia and industries due to the data richness in PDF, from plain text, tables to high resolution
images and intricate vector graphics, presenting an opportunity and a challenge at the same time.
Tr...


__Step 2__ — Build the PROMPT (combine question + chunks)

In [5]:
def build_prompt(query, chunks):
    """
    
    Combines retrieved chunks + user question into a prompt for the LLM.
    
    """
    
    context = "\n\n".join(chunks)
    
    prompt = f"""Answer the question based only on the context below. 
If the context doesn't contain enough information to answer, say "I don't have enough information to answer this."

Context:
{context}

Question: {query}

Answer:"""
    
    return prompt

# Test it
test_prompt = build_prompt(
    "What challenges does this paper address with PDFs?",
    test_chunks
)
print(test_prompt)

Answer the question based only on the context below. 
If the context doesn't contain enough information to answer, say "I don't have enough information to answer this."

Context:
academia and industries due to the data richness in PDF, from plain text, tables to high resolution
images and intricate vector graphics, presenting an opportunity and a challenge at the same time.
Traditional RAG-based QA systems focus primarily on text Lin [2024], Ma et al. [2023], Siriwardhana
et al. [2023] while non-textual elements such as images, charts, tables and diagrams within PDFs
are not thoroughly explored. Our objective is to address this gap by developing a comprehensive

types of content from PDFs, including text, images, and layouts. By default, it uses the ‘Surya’ OCR
engine for efficient and accurate text recognition. The tool is configured to run OCR on all pages of a
PDF, even if some text can be directly extracted, ensuring comprehensive text recognition across the
entire document.
In add

In [6]:
def generate_answer(prompt):
    """
    Sends the prompt to Groq's LLaMA model and returns the answer.
    """
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that answers questions based strictly on the provided context."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2  # low temperature = more focused, less creative
    )
    return response.choices[0].message.content

# Generate the answer using the prompt from Step 2
answer = generate_answer(test_prompt)
print("Answer:")
print(answer)

Answer:
The challenges addressed by this paper with PDFs include:

1. Non-textual elements not being thoroughly explored by traditional RAG-based QA systems.
2. The Markdown's tools performance in the conversion of complex PDF structures, which might lead to format errors or information loss.
3. Potentially unnecessary retrieval steps and latency overhead due to always retrieving for every query.


__Step 4__ — Package it all into ONE function (the full pipeline)

In [7]:
def rag_query(question, n_results=3, verbose=False):
    """
    
    Complete RAG pipeline: retrieve chunks, build prompt, generate answer.
    
    """
    
    # Step 1: Retrieve
    chunks = retrieve_chunks(question, n_results=n_results)
    
    # Step 2: Build prompt
    prompt = build_prompt(question, chunks)
    
    # Step 3: Generate
    answer = generate_answer(prompt)
    
    if verbose:
        print("--- Retrieved chunks ---")
        for i, chunk in enumerate(chunks):
            print(f"\n[{i+1}] {chunk[:150]}...")
        print("\n--- Answer ---")
    
    return answer



### Add SOURCE CITATIONS to our function

A simple, real mitigation: show users WHICH chunks were used, so they can judge for themselves.

In [16]:
def rag_query_with_sources(question, n_results=3):
    """
    RAG pipeline that also returns source chunks for verification.
    """
    chunks = retrieve_chunks(question, n_results=n_results)
    prompt = build_prompt(question, chunks)
    answer = generate_answer(prompt)
    
    return {
        "question": question,
        "answer": answer,
        "sources": chunks
    }

# Test it
result = rag_query_with_sources("What dataset did the researchers use?")
print(f"Question: {result['question']}")
print(f"\nAnswer: {result['answer']}")
print(f"\n--- Sources used (verify yourself) ---")
for i, source in enumerate(result['sources']):
    print(f"\n[Source {i+1}]: {source[:200]}...")

Question: What dataset did the researchers use?

Answer: Our dataset consists of 8 private internal documents from a production company.

--- Sources used (verify yourself) ---

[Source 1]: scalability, computational requirements, and datasets scope. A case study in the agricultural domain
Gupta et al. [2024] demonstrated the approach combining RAG and Fine-Tuning exhibited superior
perf...

[Source 2]: image reference IDs, we retrieve the corresponding image from the database and display it to the user.
In addition to handling images, if the LLM’s output includes dictionary-formatted tables, we extr...

[Source 3]: Shuqiang Zhang, Sinong Wang, Sneha Agarwal, Soji Sajuyigbe, Soumith Chintala, Stephanie
Max, Stephen Chen, Steve Kehoe, Steve Satterfield, Sudarshan Govindaprasad, Sumit Gupta,
Summer Deng, Sungmin Ch...


In [14]:
user_query = input("Query:") # Which OCR engine is mentioned for text recognition?
user_query

Query:What were the main findings or results of this paper?


'What were the main findings or results of this paper?'

In [15]:
answer = rag_query(user_query, verbose=True)
print(answer)

--- Retrieved chunks ---

[1] 2024.
Edward J Hu, Yelong Shen, Phillip Wallis, Zeyuan Allen-Zhu, Yuanzhi Li, Shean Wang, Lu Wang,
and Weizhu Chen. Lora: Low-rank adaptation of large...

[2] engine, 2023. URL https://journal.hexmos.com/marker-pdf-document-ai/ . Last
visited December 10,2023.
Hossein Rajabzadeh, Suyuchen Wang, Hyock Ju Kwon...

[3] Shuqiang Zhang, Sinong Wang, Sneha Agarwal, Soji Sajuyigbe, Soumith Chintala, Stephanie
Max, Stephen Chen, Steve Kehoe, Steve Satterfield, Sudarshan G...

--- Answer ---
I don't have enough information to answer this. The provided context lists various papers and authors but does not mention the specific paper being referred to.


__What temperature=0.2 means — quick but important concept__


- temperature = 0    → always picks the most likely next word (deterministic, factual)
- temperature = 0.2  → mostly factual, tiny bit of variation
- temperature = 1.0  → creative, varied responses (good for stories)
- temperature = 2.0  → very random, often nonsensical


Note:
- For RAG / factual Q&A → ALWAYS use LOW temperature (0-0.3)
- For creative writing  → use HIGHER temperature (0.7-1.2)

## Conclusion:

Our question: "What challenges does this paper address with PDFs?"

Our RAG System:
1. Converted question to embedding
2. Searched 93 chunks → found 3 most relevant
3. Built a prompt with those 3 chunks as context
4. Sent it to LLaMA 3.1 via Groq
5. LLaMA read ONLY your provided context
6. Generated a clean, accurate, structured answer


__RAG REDUCES hallucination__ — it does NOT eliminate it.

RAG without sources  → users trust wrong answers blindly
RAG WITH sources     → users can VERIFY and catch errors

This is why "showing sources" isn't a nice UI feature —
it's a CORE SAFETY MECHANISM.